# 第 1 周练习 – 商业洞察讲解器

本 notebook 演示一个小工具：用大语言模型讲解科技公司的商业模式。

对比以下模型的回答：
- GPT-4o-mini（OpenAI）
- 本地 Ollama 上的 Llama 3.2

目标是练习提示词设计，并比较托管模型与本地模型的回答质量。



In [ ]:
# 导入依赖
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display



In [ ]:
# 加载环境变量并初始化客户端
load_dotenv()

MODEL_GPT = "gpt-4o-mini"
MODEL_LLAMA = "llama3.2"
openrouter_url = "https://openrouter.ai/api/v1"

OLLAMA_URL = os.getenv("OLLAMA_URL", "http://localhost:11434")

# 通过 OpenRouter 访问 GPT
client = OpenAI(base_url=openrouter_url, api_key=os.getenv("OPENAI_KEY"))



In [ ]:
# 系统提示与示例问题（分析 Stripe 商业模式）
SYSTEM_PROMPT = """
You are an expert in startup and SaaS business models.

Provide clear and structured explanations that help developers
and entrepreneurs understand how technology companies operate.
"""

question = """
Analyze the business model of Stripe.

Provide:
1. Short summary
2. Revenue model
3. Target customers
4. Competitive advantage
"""



In [ ]:
def ask_gpt(question: str) -> str:
    """查询 GPT 模型并返回回复。"""

    response = client.chat.completions.create(
        model=MODEL_GPT,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
    )

    return response.choices[0].message.content



In [ ]:
# 调用 GPT 并展示
gpt_result = ask_gpt(question)

display(Markdown("## GPT-4o-mini Response"))
display(Markdown(gpt_result))



In [ ]:
def ask_llama(question: str) -> str:
    """通过 Ollama 查询本地 Llama 模型。"""

    response = requests.post(
        f"{OLLAMA_URL}/api/chat",
        json={
            "model": MODEL_LLAMA,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": question},
            ],
            "stream": False
        },
    )

    data = response.json()
    return data["message"]["content"]



In [ ]:
# 调用本地 Llama 并展示
llama_result = ask_llama(question)

display(Markdown("## Llama 3.2 Response"))
display(Markdown(llama_result))



In [ ]:
# 观察记录
print("Observation:")

print("""
GPT produced a more structured explanation and clearer breakdown of the business model.

Llama generated a shorter response but still captured the main revenue model
and core target users.

This shows how prompt structure can guide models to produce useful business insights.
""")

